In [ ]:
import numpy as np
import pandas as pd
from scipy import stats
import seaborn as sns
import matplotlib.pyplot as plt
from google.colab import files
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
import statsmodels.api as sm
from sklearn.metrics import mean_absolute_error, mean_squared_error
from google.colab import drive
from pathlib import Path

In [ ]:
!rm -rf /content/diploma
!git clone https://github.com/anastasiakatogina/diploma.git /content/diploma

In [ ]:
BASE_PATH = Path("/content/diploma/data/raw")

PATHS = {
    "v": BASE_PATH / "02_07_New_loans_housing.xlsx",
    "c": BASE_PATH / "02_10_Quantity_mortgage.xlsx",
    "z": BASE_PATH / "02_09_Debt_housing.xlsx",
    "k": BASE_PATH / "02_13_Rates_mortgage.xlsx",
    "i": BASE_PATH/ "inflation_key_rate.xlsx",
    "zp": BASE_PATH/ "salary.xls",
    "cl": BASE_PATH/ "housing_prices.xls",
}

In [ ]:
vk = pd.read_excel(PATHS["v"], sheet_name="в рублях", index_col=0, header=2)
vk = vk.T

In [ ]:
ck = pd.read_excel(PATHS["c"], sheet_name='в рублях', index_col=0, header=2) #количество кредитов шт
ck = ck.T
ck.index = vk.index

In [ ]:
zk = pd.read_excel(PATHS["z"], sheet_name='в рублях', index_col=0, header=2) #задолженность по кредитам млн
zk = zk.T
zk.index = vk.index

In [ ]:
ks = pd.read_excel(PATHS["k"], sheet_name='ставка в рублях', index_col=0, header=2) #средневзвешенная ставка по кредитам в %
ks = ks.T
ks.index = vk.index

In [ ]:
srok = pd.read_excel(PATHS["k"], sheet_name='срок в руб', index_col=0, header=2) #срок кредитования в месяцах
srok = srok.T
srok.index = vk.index

In [ ]:
ins = pd.read_excel(PATHS["i"], usecols=[0, 1, 2], index_col=0) #инфляция и ключевая ставка
ins = ins[::-1]

In [ ]:
clz = pd.read_excel(PATHS["cl"], index_col=[0, 1], header=[2, 3, 4]) #класс жилья

In [ ]:
zp_m = pd.read_excel(PATHS["zp"], index_col=[1, 2],  header=[2]) #средние зп(среднее арифметическое)
zp_m = zp_m.drop(columns=zp_m.columns[0])
zp_m["Российская Федерация"] = zp_m["Российская Федерация"].fillna(zp_m["Российская Федерация без учета новых субъектов (с 01.01.2023)"])
zp_m = zp_m.drop(columns=zp_m.columns[1])
zp_m = zp_m.iloc[24:]
zp_m.index = zp_m.index.map(lambda t: f"{str(t[1]).strip().capitalize()} {int(t[0])}")

In [ ]:
ins = ins.iloc[64:]
ins = ins.iloc[:-1]
ins.index = zp_m.index

In [ ]:
df_rf = pd.DataFrame(ins)
df_rf.columns = ['Ключ. ставка', 'Инфляция']

In [ ]:
old_n = pd.DataFrame()
old_n2 = pd.DataFrame()
old_n2['Количество'] = ck['РОССИЙСКАЯ ФЕДЕРАЦИЯ'].iloc[:-1]
old_n2 = pd.concat([old_n, old_n2], axis=0)
old_n2.index = df_rf.index
df_rf['Количество'] = old_n2['Количество']

In [ ]:
old_n = pd.DataFrame()
old_n2 = pd.DataFrame()
old_n2['Задолженность'] = zk['РОССИЙСКАЯ ФЕДЕРАЦИЯ'].iloc[:-1]
old_n3 = pd.DataFrame()
old_n3['Задолженность'] = pd.concat([old_n, old_n2], axis=0)
old_n3.index = df_rf.index
df_rf['Задолженность'] = old_n3['Задолженность']

In [ ]:
old_n = pd.DataFrame()
old_n2 = pd.DataFrame()
old_n2['Объем'] = vk['РОССИЙСКАЯ ФЕДЕРАЦИЯ'].iloc[:-1]
old_n3 = pd.DataFrame()
old_n3['Объем'] = pd.concat([old_n, old_n2], axis=0)
old_n3.index = df_rf.index
df_rf['Объем'] = old_n3['Объем']

In [ ]:
old_n = pd.DataFrame()
old_n2 = pd.DataFrame()
old_n2['Ставка'] = ks['РОССИЙСКАЯ ФЕДЕРАЦИЯ'].iloc[:-1]
old_n3 = pd.DataFrame()
old_n3['Ставка'] = pd.concat([old_n, old_n2], axis=0)
old_n3.index = df_rf.index
df_rf['Ставка'] = old_n3['Ставка']

In [ ]:
old_n = pd.DataFrame()
old_n2 = pd.DataFrame()
old_n2['Срок'] = srok['РОССИЙСКАЯ ФЕДЕРАЦИЯ'].iloc[:-1]
old_n3 = pd.DataFrame()
old_n3['Срок'] = pd.concat([old_n, old_n2], axis=0)
old_n3.index = df_rf.index
df_rf['Срок'] = old_n3['Срок']

In [ ]:
df_rf = df_rf.join(zp_m['Российская Федерация']).rename(columns={'Российская Федерация': 'Средняя ЗП'})

In [ ]:
df_rf['Средняя сумма кредита'] = df_rf['Объем'].div(df_rf['Количество'], fill_value=1)

In [ ]:
display(df_rf)

In [ ]:
housing = pd.DataFrame()
clz = clz['Российская Федерация']
housing['low'] = clz['Низкого качества'].mean(axis=1)
housing['typical'] = clz['Квартиры среднего качества (типовые)'].mean(axis=1)
housing['improved'] = clz['Улучшенного качества'].mean(axis=1)
housing['elite'] = clz['Элитные квартиры'].mean(axis=1)
housing['Средняя цена жилья'] = housing.mean(axis=1)
housing = housing.reset_index()
def quarter_to_month(q):
    q = str(q).strip()
    if q == 'I квартал':
        return 'Март'
    elif q == 'II квартал':
        return 'Июнь'
    elif q == 'III квартал':
        return 'Сентябрь'
    elif q == 'IV квартал':
        return 'Декабрь'

housing['date'] = housing['level_1'].apply(quarter_to_month) + ' ' + housing['level_0'].astype(str)
housing = housing.set_index('date')
housing = housing.drop(columns=['level_0', 'level_1'])
housing = housing[['low', 'typical', 'improved', 'elite', 'Средняя цена жилья']]
month_map = {
    'Январь': '01', 'Февраль': '02', 'Март': '03', 'Апрель': '04',
    'Май': '05', 'Июнь': '06', 'Июль': '07', 'Август': '08',
    'Сентябрь': '09', 'Октябрь': '10', 'Ноябрь': '11', 'Декабрь': '12'
}

housing_dt = housing.copy()

housing_dt.index = [
    pd.to_datetime(f"{idx.split()[1]}-{month_map[idx.split()[0]]}-01")
    for idx in housing_dt.index
]

housing_dt.head()
housing_monthly = housing_dt.resample('MS').interpolate()
housing_monthly = housing_monthly.loc['2013-08':'2025-06']
ru_months = {
    '01': 'Январь', '02': 'Февраль', '03': 'Март', '04': 'Апрель',
    '05': 'Май', '06': 'Июнь', '07': 'Июль', '08': 'Август',
    '09': 'Сентябрь', '10': 'Октябрь', '11': 'Ноябрь', '12': 'Декабрь'
}

housing_monthly.index = [
    f"{ru_months[d.strftime('%m')]} {d.strftime('%Y')}"
    for d in housing_monthly.index
]
housing_monthly = housing_monthly.round()
housing_monthly.head(12)

In [ ]:
df_full = df_rf.join(housing_monthly, how='left')
plt.figure(figsize=(10,8))
sns.heatmap(df_full.corr(numeric_only=True), annot=True, cmap='coolwarm')
plt.show()

тут я решаю избавиться от мультиколлиниарности удалением. Возможно попробую еще метод главных компонент

In [ ]:
model_df = df_full[[
    'Задолженность',
    'Ключ. ставка',
    'Инфляция',
    'Ставка',
    'Срок',
    'Средняя ЗП',
    'Средняя сумма кредита',
    'Средняя цена жилья'
]].copy()
model_df = model_df.rename(columns={'price_mean': 'Средняя цена жилья'})
model_df = model_df.loc['Январь 2019':]
model_df.head()

In [ ]:
model_df['Средняя ЗП'] = model_df['Средняя ЗП'].rolling(12).mean()
model_df = model_df.fillna(model_df['Февраль 2020' : 'Январь 2021'].mean())
model_df = model_df.rename(columns={'price_mean': 'Средняя цена жилья'})

In [ ]:
model_df.to_csv("model_df.csv")
files.download("model_df.csv")

In [ ]:
model_df.describe()

In [ ]:
cols = [
    'Задолженность',
    'Ключ. ставка',
    'Инфляция',
    'Ставка',
    'Срок',
    'Средняя ЗП',
    'Средняя сумма кредита',
    'Средняя цена жилья'
]

for col in cols:
    plt.figure(figsize=(20,5))

    plt.plot(model_df.index, model_df[col], linewidth=2)

    plt.title(f'Динамика показателя: {col}', fontsize=14)
    plt.xlabel('Период')
    plt.ylabel(col)

    plt.xticks(rotation=90)
    plt.grid(True)

    plt.tight_layout()
    plt.show()
    plt.savefig("graph.png", dpi=300, bbox_inches='tight')
    plt.close()

In [ ]:
cols = [
    'Задолженность',
    'Ключ. ставка',
    'Инфляция',
    'Ставка',
    'Срок',
    'Средняя сумма кредита',
    'Средняя цена жилья'
]

X = model_df[cols].copy()
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

kmeans = KMeans(
    n_clusters=5,
    random_state=42,
    n_init=10
)

model_df['кластер'] = kmeans.fit_predict(X_scaled)
cluster_table = model_df.groupby('кластер')[cols].mean()
cluster_table = cluster_table.round(0).astype("Int64")
cluster_table

In [ ]:
for c in sorted(model_df['кластер'].dropna().unique()):
    print(f'\nКластер {c}')
    dates = model_df[model_df['кластер'] == c].index
    print(dates)

In [ ]:
X = model_df[cols].copy()
X = X.dropna()

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

inertia = []

K = range(1,10)

for k in K:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans.fit(X_scaled)
    inertia.append(kmeans.inertia_)

plt.figure(figsize=(6,4))
plt.plot(K, inertia, marker='o')
plt.xlabel("Количество кластеров k")
plt.ylabel("Inertia")
plt.title("Метод локтя")
plt.show()

In [ ]:
model_df.groupby('кластер').mean().round(2)
model_df.groupby('кластер').std().round(2)

In [ ]:
features = [
    'Ключ. ставка',
    'Инфляция',
    'Ставка',
    'Срок',
    'Средняя ЗП',
    'Средняя сумма кредита',
    'Средняя цена жилья'
]

X = model_df[features].copy()
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
pca = PCA()
X_pca = pca.fit_transform(X_scaled)

pca.explained_variance_ratio_

In [ ]:
plt.figure(figsize=(8,5))

plt.plot(
    range(1, len(pca.explained_variance_ratio_) + 1),
    pca.explained_variance_ratio_.cumsum(),
    marker='o'
)

plt.xlabel('Количество компонент')
plt.ylabel('Накопленная объяснённая дисперсия')
plt.title('Выбор числа главных компонент')
plt.grid(True)

plt.show()

In [ ]:
pca = PCA(n_components=3)
X_pca = pca.fit_transform(X_scaled)
pca_df = pd.DataFrame(
    X_pca,
    columns=['PC1', 'PC2', 'PC3'],
    index=model_df.index
)
pca_df = pca_df.round(2)
pca_df.head()

In [ ]:
X = pca_df[['PC1', 'PC2', 'PC3']]
y = model_df['Задолженность']
X = sm.add_constant(X)
model = sm.OLS(y, X).fit()
print(model.summary())

In [ ]:
n_sim = 10000
np.random.seed(42)
pc1_sim = np.random.normal(
    pca_df['PC1'].mean(),
    pca_df['PC1'].std(),
    n_sim
)

pc2_sim = np.random.normal(
    pca_df['PC2'].mean(),
    pca_df['PC2'].std(),
    n_sim
)

pc3_sim = np.random.normal(
    pca_df['PC3'].mean(),
    pca_df['PC3'].std(),
    n_sim
)
sim_df = pd.DataFrame({
    'PC1': pc1_sim,
    'PC2': pc2_sim,
    'PC3': pc3_sim
})

sim_df = sim_df.round(2)
sim_df.head()

In [ ]:
sim_df = sm.add_constant(sim_df)
sim_df['Задолженность_модель'] = model.predict(sim_df)
sim_df['Задолженность_модель'] = sim_df['Задолженность_модель'].round(0).astype("Int64")

sim_df.head()

In [ ]:
plt.figure(figsize=(10,5))
plt.hist(sim_df['Задолженность_модель'], bins=40)

plt.title('Имитационное распределение задолженности')
plt.xlabel('Задолженность')
plt.ylabel('Частота')

plt.grid(True)
plt.show()

In [ ]:
np.percentile(sim_df['Задолженность_модель'], [5, 50, 95]).round(2)

In [ ]:
features = [
    'Ключ. ставка',
    'Инфляция',
    'Ставка',
    'Срок',
    'Средняя ЗП',
    'Средняя сумма кредита',
    'Средняя цена жилья'
]

df = model_df.copy()
df['кластер'].value_counts()

In [ ]:
cluster_stats = {}
for c in df['кластер'].unique():
    temp = df[df['кластер'] == c]
    stats = {}
    for col in features:
        stats[col] = {
            'mean': temp[col].mean(),
            'std': temp[col].std()
        }

    cluster_stats[c] = stats
results = {}
n_sim = 3000

In [ ]:
for c in cluster_stats:
    sim_data = pd.DataFrame()
    for col in features:
        mean = cluster_stats[c][col]['mean']
        std = cluster_stats[c][col]['std']
        sim_data[col] = np.random.normal(mean, std, n_sim)
    sim_scaled = scaler.transform(sim_data)
    sim_pca = pca.transform(sim_scaled)

    sim_pca_df = pd.DataFrame(
        sim_pca,
        columns=['PC1', 'PC2', 'PC3']
    )
    sim_pca_df = sm.add_constant(sim_pca_df)
    sim_pca_df['Задолженность'] = model.predict(sim_pca_df)
    results[c] = sim_pca_df['Задолженность']

In [ ]:
plt.figure(figsize=(12,6))

for c in results:
    plt.hist(results[c], bins=40, alpha=0.5, label=f'Кластер {c}')
plt.legend()
plt.title('Распределение задолженности по кластерам')
plt.xlabel('Задолженность')
plt.ylabel('Частота')
plt.grid(True)

plt.show()

In [ ]:
cluster_means = model_df.groupby('кластер')[
    [
        'Задолженность',
        'Ключ. ставка',
        'Инфляция',
        'Ставка',
        'Срок',
        'Средняя ЗП',
        'Средняя сумма кредита',
        'Средняя цена жилья'
    ]
].mean()
cluster_means.round(2)
cluster_means

In [ ]:
cluster_names = {
    0: 'Низкая активность',
    1: 'Рост рынка',
    2: 'Кризис / спад',
    3: 'Стабильный рынок',
    4: 'Перегрев / пик'
}

In [ ]:
var_results = {}
for c in results:

    var_5 = np.percentile(results[c], 5)
    var_1 = np.percentile(results[c], 1)

    var_results[c] = {
        'VaR_5%': var_5,
        'VaR_1%': var_1
    }

In [ ]:
worst_case = {}
for c in results:
    worst = results[c].min()
    worst_case[c] = worst
worst_case

In [ ]:
cluster_probs = model_df['кластер'].value_counts(normalize=True)
cluster_probs.index = cluster_probs.index.map(cluster_names)
cluster_probs

In [ ]:
weighted_results = []
for c in results:
    prob = cluster_probs[c]
    sample = results[c].sample(
        int(len(results[c]) * prob),
        replace=True
    )
    weighted_results.append(sample)
weighted_results = pd.concat(weighted_results)

In [ ]:
y_true = y
y_pred = model.predict(X)

print('MAE =', mean_absolute_error(y_true, y_pred))
print('RMSE =', np.sqrt(mean_squared_error(y_true, y_pred)))

In [ ]:
scenario_stats = {}

for c in cluster_stats:
    scenario_stats[c] = {}

    for col in features:
        mean = cluster_stats[c][col]['mean']
        std = cluster_stats[c][col]['std']

        scenario_stats[c][col] = {
            'base_mean': mean,
            'optimistic_mean': mean * 0.95 if col in ['Ключ. ставка', 'Инфляция', 'Ставка'] else mean * 1.05,
            'pessimistic_mean': mean * 1.05 if col in ['Ключ. ставка', 'Инфляция', 'Ставка'] else mean * 0.95,
            'std': std
        }

In [ ]:
weighted_results = []
cluster_order = sorted(model_df['кластер'].dropna().unique())
target_n = 10000

for c in cluster_order:
    sim_values = pd.Series(results[c]).dropna()
    prob = cluster_probs[c]
    n_c = max(1, int(target_n * prob))

    sample_c = sim_values.sample(n=n_c, replace=True, random_state=42)
    weighted_results.append(sample_c)

weighted_results = pd.concat(weighted_results, ignore_index=True)

overall_mean = weighted_results.mean()
overall_median = weighted_results.median()
overall_var_5 = np.percentile(weighted_results, 5)
overall_var_1 = np.percentile(weighted_results, 1)
overall_worst = weighted_results.min()
overall_cvar_5 = weighted_results[weighted_results <= overall_var_5].mean()

overall_stats = pd.DataFrame({
    'Показатель': [
        'Среднее',
        'Медиана',
        'VaR 5%',
        'VaR 1%',
        'CVaR 5%',
        'Worst case'
    ],
    'Значение': [
        overall_mean,
        overall_median,
        overall_var_5,
        overall_var_1,
        overall_cvar_5,
        overall_worst
    ]
})

overall_stats['Значение'] = overall_stats['Значение'].round(0).astype('Int64')

overall_stats

In [ ]:
plt.figure(figsize=(10, 5))
plt.hist(weighted_results, bins=40)

plt.title('Итоговое распределение задолженности по всем режимам')
plt.xlabel('Задолженность, млн руб.')
plt.ylabel('Частота')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
np.random.seed(42)

cluster_names = {
    0: 'Низкая активность',
    1: 'Рост рынка',
    2: 'Кризис / спад',
    3: 'Стабильный рынок',
    4: 'Перегрев / пик'
}

cluster_probs = (
    model_df['кластер']
    .value_counts(normalize=True)
    .reindex(cluster_order)
    .fillna(0)
)

risk_summary = []

for c in cluster_order:
    sim_values = pd.Series(results[c]).dropna()

    mean_val = sim_values.mean()
    median_val = sim_values.median()
    var_5 = np.percentile(sim_values, 5)
    var_1 = np.percentile(sim_values, 1)
    worst_case = sim_values.min()
    cvar_5 = sim_values[sim_values <= var_5].mean()

    risk_summary.append({
        'Кластер': c,
        'Режим': cluster_names[c],
        'Вероятность': cluster_probs[c],
        'Средняя задолженность': mean_val,
        'Медиана': median_val,
        'VaR 5%': var_5,
        'VaR 1%': var_1,
        'CVaR 5%': cvar_5,
        'Worst case': worst_case
    })

risk_table = pd.DataFrame(risk_summary)

numeric_cols = [
    'Средняя задолженность',
    'Медиана',
    'VaR 5%',
    'VaR 1%',
    'CVaR 5%',
    'Worst case'
]

risk_table[numeric_cols] = risk_table[numeric_cols].round(0).astype('Int64')
risk_table['Вероятность'] = (risk_table['Вероятность'] * 100).round(1)

risk_table = risk_table.sort_values('Кластер').reset_index(drop=True)

risk_table

In [ ]:
plot_df = risk_table.copy().set_index('Режим')

ax = plot_df[['VaR 5%', 'VaR 1%', 'Worst case']].plot(
    kind='bar',
    figsize=(10, 5)
)

ax.set_title('Сравнение риск-метрик по экономическим режимам')
ax.set_xlabel('Экономический режим')
ax.set_ylabel('Задолженность, млн руб.')
plt.xticks(rotation=20, ha='right')
plt.grid(True, axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
risk_table

In [ ]:
q33 = risk_table['VaR 5%'].quantile(0.33)
q66 = risk_table['VaR 5%'].quantile(0.66)

def risk_level(x):
    if x >= q66:
        return 'Высокий'
    elif x >= q33:
        return 'Средний'
    return 'Умеренный'

risk_table['Уровень риска'] = risk_table['VaR 5%'].apply(risk_level)

risk_table

In [ ]:
import ipywidgets as widgets
from IPython.display import display

In [ ]:
cluster_order = [0, 1, 2, 3, 4]

cluster_names = {
    0: 'Низкая активность',
    1: 'Рост рынка',
    2: 'Кризис / спад',
    3: 'Стабильный рынок',
    4: 'Перегрев / пик'
}

decision_table = pd.DataFrame({
    'Кластер': cluster_order
})

decision_table['Режим'] = decision_table['Кластер'].map(cluster_names)

mean_debt = (
    model_df.groupby('кластер')['Задолженность']
    .mean()
    .reindex(cluster_order)
    .round(0)
    .astype('Int64')
    .values
)

var_5 = (
    pd.Series({c: np.percentile(results[c], 5) for c in results})
    .reindex(cluster_order)
    .round(0)
    .astype('Int64')
    .values
)

worst_case = (
    pd.Series({c: np.min(results[c]) for c in results})
    .reindex(cluster_order)
    .round(0)
    .astype('Int64')
    .values
)

cluster_probs = (
    model_df['кластер']
    .value_counts(normalize=True)
    .reindex(cluster_order)
    .fillna(0)
)

decision_table['Вероятность'] = cluster_probs.values
decision_table['Средняя задолженность'] = mean_debt
decision_table['VaR 5%'] = var_5
decision_table['Worst case'] = worst_case

q33 = decision_table['VaR 5%'].quantile(0.33)
q66 = decision_table['VaR 5%'].quantile(0.66)

def risk_level(var_value):
    if var_value >= q66:
        return 'Высокий'
    elif var_value >= q33:
        return 'Средний'
    return 'Умеренный'

decision_table['Уровень риска'] = decision_table['VaR 5%'].apply(risk_level)

scenario_widget = widgets.Dropdown(
    options=['Базовый', 'Оптимистичный', 'Пессимистичный'],
    value='Базовый',
    description='Сценарий:'
)

cluster_widget = widgets.Dropdown(
    options=['Все'] + [cluster_names[c] for c in cluster_order],
    value='Все',
    description='Режим:'
)

stress_widget = widgets.IntSlider(
    value=10,
    min=0,
    max=30,
    step=5,
    description='Стресс %:'
)

def apply_scenario(df, scenario, stress):
    df = df.copy()
    factor = stress / 100

    if scenario == 'Оптимистичный':
        df['Средняя задолженность'] = df['Средняя задолженность'] * (1 - factor * 0.20)
        df['VaR 5%'] = df['VaR 5%'] * (1 - factor * 0.25)
        df['Worst case'] = df['Worst case'] * (1 - factor * 0.35)

    elif scenario == 'Пессимистичный':
        df['Средняя задолженность'] = df['Средняя задолженность'] * (1 + factor * 0.20)
        df['VaR 5%'] = df['VaR 5%'] * (1 + factor * 0.25)
        df['Worst case'] = df['Worst case'] * (1 + factor * 0.35)

    df['Средняя задолженность'] = df['Средняя задолженность'].round(0).astype('Int64')
    df['VaR 5%'] = df['VaR 5%'].round(0).astype('Int64')
    df['Worst case'] = df['Worst case'].round(0).astype('Int64')

    q33_s = df['VaR 5%'].quantile(0.33)
    q66_s = df['VaR 5%'].quantile(0.66)

    def risk_level_s(var_value):
        if var_value >= q66_s:
            return 'Высокий'
        elif var_value >= q33_s:
            return 'Средний'
        return 'Умеренный'

    df['Уровень риска'] = df['VaR 5%'].apply(risk_level_s)

    return df

def update_dashboard(scenario, cluster, stress):
    df = decision_table.copy()

    if cluster != 'Все':
        df = df[df['Режим'] == cluster]

    df = apply_scenario(df, scenario, stress)

    show_df = df.copy()
    show_df['Вероятность'] = (show_df['Вероятность'] * 100).round(1).astype(str) + '%'
    show_df = show_df.set_index('Режим')

    display(
        show_df.style.format({
            'Кластер': '{:.0f}',
            'Средняя задолженность': '{:,.0f}',
            'VaR 5%': '{:,.0f}',
            'Worst case': '{:,.0f}'
        })
    )

    fig, ax = plt.subplots(figsize=(10, 4.8))
    bars = ax.bar(df['Режим'], df['VaR 5%'])

    ax.set_title(f'VaR 5% по экономическим режимам — {scenario}')
    ax.set_xlabel('Экономический режим')
    ax.set_ylabel('Задолженность, млн руб.')

    y_max = max(df['VaR 5%']) * 1.15
    ax.set_ylim(0, y_max)
    ax.set_yticks(np.linspace(0, y_max, 6))
    ax.set_yticklabels([f'{v/1_000_000:.1f}' for v in np.linspace(0, y_max, 6)])

    for bar, value in zip(bars, df['VaR 5%']):
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() + y_max * 0.015,
            f'{value/1_000_000:.1f}',
            ha='center',
            va='bottom',
            fontsize=10
        )

    plt.xticks(rotation=18, ha='right')
    plt.grid(True, axis='y', alpha=0.3)
    plt.tight_layout()
    plt.show()

widgets.interactive(
    update_dashboard,
    scenario=scenario_widget,
    cluster=cluster_widget,
    stress=stress_widget
)